In [6]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from neuromancer.system import Node, System
from neuromancer.dynamics import integrators, ode
from neuromancer.modules import blocks
from neuromancer.dataset import DictDataset

In [13]:
trainX[:, 0, :]

array([[0.3061896 , 0.86351029]])

In [ ]:
def get_data(sys, nsim, nsteps, ts, bs):
    """
    :param nsteps: (int) Number of timesteps for each batch of training data
    :param sys: (psl.system)
    :param ts: (float) step size
    :param bs: (int) batch size

    """

    trainX          = np.load("../data/trainX.npy")
    output_train    = np.load("../data/output_train.npy")
    validateX       = np.load("../data/validateX.npy")
    output_validate = np.load("../data/output_validate.npy")
    testX           = np.load("../data/testX.npy")
    output_test     = np.load("../data/output_test.npy")

    train_data = DictDataset({'X': trainX, 'xn': trainX[:, 0:1, :],
                              'U': output_train}, name='train')


    train_sim, dev_sim, test_sim = [sys.simulate(nsim=nsim, ts=ts) for i in range(3)]
    nx = sys.nx
    nbatch = nsim//nsteps
    length = (nsim//nsteps) * nsteps

    trainX = train_sim['X'][:length].reshape(nbatch, nsteps, nx)
    trainX = torch.tensor(trainX, dtype=torch.float32)
    train_data = DictDataset({'X': trainX, 'xn': trainX[:, 0:1, :]}, name='train')
    train_loader = DataLoader(train_data, batch_size=bs,
                              collate_fn=train_data.collate_fn, shuffle=True)

    devX = dev_sim['X'][:length].reshape(nbatch, nsteps, nx)
    devX = torch.tensor(devX, dtype=torch.float32)
    dev_data = DictDataset({'X': devX, 'xn': devX[:, 0:1, :]}, name='dev')
    dev_loader = DataLoader(dev_data, batch_size=bs,
                            collate_fn=dev_data.collate_fn, shuffle=True)

    testX = test_sim['X'][:length].reshape(1, nsim, nx)
    testX = torch.tensor(testX, dtype=torch.float32)
    test_data = {'X': testX, 'xn': testX[:, 0:1, :]}

    return train_loader, dev_loader, test_data


construct a continuous-time NODE model $\dot{x}=f_\theta(x)$ with trainable parameters $\theta$
.

In [ ]:
# define neural network of the NODE
fx = blocks.MLP(nx, nx, bias=True,
                 linear_map=torch.nn.Linear,
                 nonlin=torch.nn.ReLU,
                 hsizes=[60, 60, 60])

# integrate NODE with adjoint-based solver
fxRK4 = integrators.DiffEqIntegrator(fx, h=ts, method='rk4')

# create symbolic system model in Neuromancer
model = Node(fxRK4, ['xn'], ['xn'], name='NODE')
dynamics_model = System([model], name='system', nsteps=nsteps)